# Film Expert AI Assistant

## Locate RAG documents

This notebook prefers local files in `../data/rag/` (relative to `notebooks/`).
Google Drive is used only as a fallback if those files are not present (Colab).

In [48]:
from pathlib import Path

# Prefer local RAG documents; mount Google Drive only if they are missing.
candidates = [
    Path.cwd() / "data" / "rag",
    Path.cwd().parent / "data" / "rag",
    Path("/content/drive/MyDrive/data_of_RAG"),
]

DATA_DIR = next((p for p in candidates if p.is_dir()), None)

if DATA_DIR is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = Path("/content/drive/MyDrive/data_of_RAG")
    except ImportError:
        raise FileNotFoundError(
            "Could not find data/rag. Run this notebook from the project root or notebooks/ directory."
        )

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"RAG data directory not found: {DATA_DIR}")

print(f"Using RAG data directory: {DATA_DIR.resolve()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Loading API keys

Do **not** hardcode credentials in this notebook.

Set these environment variables before running:
- `OPENAI_API_KEY`
- `TMDB_API_KEY`

Example (terminal):
```bash
export OPENAI_API_KEY="your-openai-key"
export TMDB_API_KEY="your-tmdb-key"
```

In [49]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set. Export it in your environment before running this notebook.")
if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY is not set. Export it in your environment before running this notebook.")

## Installation OpenAI

In [50]:
# installation (also listed in requirements.txt)
!pip install openai requests --quiet

## Importing OpenAI libraries and initialising the client

In [51]:
from openai import OpenAI
import time  # wait for finished

client = OpenAI(api_key=OPENAI_API_KEY)

## External Film API Calls

### About API Information
- API Name: The Movie DB (TMDB) [Free version]
- API Webpage: https://developer.themoviedb.org/docs

This notebook reads the TMDB API key from the `TMDB_API_KEY` environment variable.
Do not paste API keys, bearer tokens, or account IDs into this notebook.

### import package

In [52]:
import json, requests, urllib
from pathlib import Path
from datetime import datetime

### Funciton of get data from API



In [53]:
# Prefix API URLs
api_prefix = "https://api.themoviedb.org/3"

# Define a funciton of get data of API
def fetch(endpoint, params={}):
    if not params:
        params = {}
    # construct the url
    url = f"{api_prefix}{endpoint}"
    if params:
        url = f"{url}?{urllib.parse.urlencode(params)}"
    print(f"Fetching %s = {url}")
    # fetch the page
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
          print(e)  # catch err
          return
    try:
        data = response.json()
        # print(f"response data = {data}")
        return data
    except ValueError as e:
        print(e)
        return

### Find film ID through film name

In [54]:
# Prefix for API URLs
api_prefix = "https://api.themoviedb.org/3"

# use film name to find their ID
def search_movie_id_tmdb(movie_title):
    print(f"Searching for film ID for title: {movie_title}")
    endpoint = "/search/movie"
    params = {
        "api_key": TMDB_API_KEY,  # TMDB API Key
        "query": movie_title,
        "include_adult": True,
    }
    search_results = fetch(endpoint, params=params)

    if search_results and search_results.get('results'):
        # return first ID and title + year
        first_result = search_results['results'][0]
        movie_id = first_result.get('id')
        title = first_result.get('title')
        release_date = first_result.get('release_date')
        year = release_date.split('-')[0] if release_date else ""
        print(f"film ID: {movie_id}, film name: '{title}, film year: ({year})'")
        return {
            "id": movie_id,
            "title": title,
            "year": year,
            "original_title": first_result.get('original_title')
        }
    else:
        print(f"No film found for title: {movie_title}")
        return None


### Get film information from TMDB

In [55]:
# Get film details
# This function can be called by assistant.
# For example: Please tell us about Decision to leave
def get_movie_details_tmdb(movie_title):
    # Fetches detailed information for a movie by title using TMDB.
    # Returns a formatted string summarizing key details.
    print(f"Assistant called get_movie_details_tmdb for '{movie_title}'")
    # get film ID
    movie_search_info = search_movie_id_tmdb(movie_title)

    if not movie_search_info or not movie_search_info.get('id'):
        return json.dumps({"error": f"not found movie: {movie_title}"})

    movie_id = movie_search_info['id']
    # use id to get information
    endpoint = f"/movie/{movie_id}"
    params = {
        "api_key": TMDB_API_KEY,
    }
    movie_details = fetch(endpoint, params=params)

    if movie_details:
        # the information I want got
        title = movie_details.get('title')
        original_title = movie_details.get('original_title')
        release_date = movie_details.get('release_date')
        genres = ", ".join([g['name'] for g in movie_details.get('genres', [])])
        overview = movie_details.get('overview')
        vote_average = movie_details.get('vote_average')
        vote_count = movie_details.get('vote_count')
        runtime = movie_details.get('runtime')

        # get director information
        credits_endpoint = f"/movie/{movie_id}/credits"
        credits_params = {"api_key": TMDB_API_KEY}
        credits_data = fetch(credits_endpoint, params=credits_params)
        director = "Unknown"
        if credits_data and credits_data.get('crew'):
            for crew_member in credits_data['crew']:
                if crew_member.get('job') == 'Director':
                    director = crew_member.get('name')
                    break

        # Formatted output
        details_summary = f"film name: {title} ({original_title})\n"
        details_summary += f"director: {director}\n"
        details_summary += f"release date: {release_date}\n"
        details_summary += f"type: {genres}\n"
        if runtime: details_summary += f":runtime: {runtime} min\n"
        if vote_average and vote_count:
             details_summary += f"TMDB vote: {vote_average:.1f}/10 ({vote_count} 票)\n"
        details_summary += f"\nplot sommary:\n{overview}\n"

        print(f"Fetched details for '{title}'")
        return json.dumps({"details": details_summary}) # return for json

    else:
        return json.dumps({"error": f"can't get information of film: {movie_title}"})


## Define available function mappings for function calls.

In [56]:
available_functions = {
    "get_movie_details_tmdb": get_movie_details_tmdb,
}

## Create Assistant

In [57]:
assistant = client.beta.assistants.create(
    name="Film Expert AI Assistant",
    instructions="You are a professional film critic who focuses on in-depth analyses of all aspects of cinema, including narrative, character analysis, cinematography, use of music, film themes, historical context and emotional resonance. Your reviewing style is objective and insightful. Always use file_search to query the vector store first; if nothing is found or if up-to-date information is required—then fall back to TMDB.", # System commands, defining roles and behaviours
    model="gpt-4o-mini",
    tools=[
        # Enable File Search Tool for RAG
        {"type": "file_search"},
        # define function for RAG
        { "type": "function",
          "function": {
          "name": "get_movie_details_tmdb",
          "description": "Gets the details of the specified film, including director, starring actor, genre, synopsis, rating, and so on. If also need to discuss the details of a specific film, call this function to get the information.",
          "parameters": {
                "type": "object",
                "properties": {
                    "movie_title": {
                        "type": "string",
                        "description": "The name of the film for query film details."
                    }
                },
                "required": ["movie_title"]
            }
        }
    }
    ]
)


In [58]:
print(f"assistant ID is: {assistant.id}")

assistant ID is: asst_N1ZfjSzYrkv7nWQNbe1jZ0R4


- check assistant.name

In [59]:
print(f"assistant name is: {assistant.name}")

assistant name is: Film Expert AI Assistant


## Upload and attach files for RAG
### Data path

- Film data files are stored locally in `data/rag/` (from `notebooks/`, that is `../data/rag/`).
- Google Drive path `/content/drive/MyDrive/data_of_RAG/` is only a fallback in Colab.

### Data sources
- https://www.avclub.com/park-chan-wook-on-blending-genres-and-why-revenge-is-me-1798253527
- https://www.wikiwand.com/en/articles/The_Handmaiden
- https://www.kpbs.org/news/arts-culture/2013/03/08/review-sympathy-mr-vengeance

- https://thatshelf.com/interview-chan-wook-park/
- https://www.wikiwand.com/en/articles/Stoker_(film)

- ​​https://www.bcheights.com/2022/11/02/decision-to-leave-review/#:~:text=By%20purposefully%20avoiding%20the%20phrase%2C,undertake%20and%20describe%20in%20words
- https://www.thecrimson.com/article/2022/6/3/decision-to-leave-park-chan-wook-cannes-film-festival-2022-review/#:~:text=Throughout%20the%20film%2C%20these%20kinds,into%20vivid%20clarity%20during%20their
- https://letterboxd.com/journal/between-the-lines-park-chan-wook-decision-to-leave/#:~:text=,which%20we%20will%20spare%20readers
- https://en.cinemore.jp/jp/erudition/2001/article_2002_p2.html#:~:text=The%20key%20point%20of%20,and%20Hideko%20gradually%20becomes%20clear
- https://mubi.com/en/notebook/posts/combining-the-past-and-the-future-a-conversation-with-park-chan-wook
- https://www.wikiwand.com/en/articles/Decision_to_Leave


In [60]:
# the path of the film file
data_folder_path = str(DATA_DIR) + "/"
file_paths_in_drive = [
    data_folder_path + 'Park_Chan-wook.md',
    data_folder_path + 'Decision_to_leave.md',
    data_folder_path + 'The_Handmaiden.md',
    data_folder_path + 'Stoker.md',
    data_folder_path + 'Film_noir.md',
]

# file_streams = [open(path, "rb") for path in file_paths_in_drive]

# Creating Vector Stores
print("Creating Vector Store")
vector_store = client.vector_stores.create(name="Film Knowledge Base")
print(f"Vector Store created with ID: {vector_store.id}")


# Upload files one by one to OpenAI and collect their IDs, which will be used for vector storage
file_ids = [] # stores id
print("Uploading files to OpenAI...")
for file_path in file_paths_in_drive:
    try:
        with open(file_path, "rb") as f:
            # To upload a file,  purpose='assistants'
            # client.files
            openai_file = client.files.create(file=f, purpose='assistants')
            file_ids.append(openai_file.id)
            print(f"Successfully uploaded {file_path} with ID: {openai_file.id}")
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}. Skipping.")
    except Exception as e:
        print(f"Error uploading {file_path}: {e}. Skipping.")

# Create a batch of files and add the uploaded files to the vector store
print(f"Creating file batch for Vector Store {vector_store.id} with {len(file_ids)} uploaded files.")
if file_ids:
    try:
        file_batch = client.vector_stores.file_batches.create(
            vector_store_id=vector_store.id,
            file_ids=file_ids
        )
        print(f"File Batch created with ID: {file_batch.id}, status: {file_batch.status}")

        print("Waiting for file batch processing to complete")
        while file_batch.status in ['queued', 'in_progress', 'cancelling']:
            time.sleep(1)
            file_batch = client.vector_stores.file_batches.retrieve(
                vector_store_id=vector_store.id,
                batch_id=file_batch.id
            )
            print(f"File batch status: {file_batch.status}")

        if file_batch.status == 'completed':
            print('File batch processed successfully. Files are now indexed in the vector store.')
        else:
            print(f'File batch processing failed with status: {file_batch.status}')

    except Exception as e:
        print(f"Error creating or processing file batch: {e}")

else:
    print("No file IDs collected. Skipping file batch creation.")

print(f"Assistant ID: {assistant.id}")
# Attaching Vector Stores to Helper's Tool Resources
assistant_id = assistant.id
if vector_store.id:
    print(f"Attaching Vector Store {vector_store.id} to assistant {assistant.id}")
    try:
        updated_assistant = client.beta.assistants.update(
            assistant_id,
            tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
        )
        print(f"Assistant {assistant.id} has been updated with vector store {vector_store.id} attached for file search.")
    except Exception as e:
        print(f"Error attaching vector store to assistant: {e}")
else:
    print("Vector store was not created or its ID is missing. Cannot attach to assistant.")


Creating Vector Store
Vector Store created with ID: vs_6816b16854ec8191b6b5483ee0a6f8d9
Uploading files to OpenAI...
Successfully uploaded /content/drive/MyDrive/data_of_RAG/Park_Chan-wook.md with ID: file-Q9FPRWd5xuQyFTFa2QzcVp
Successfully uploaded /content/drive/MyDrive/data_of_RAG/Decision_to_leave.md with ID: file-8ZuQbpwU9jHE8rZeY2KiJM
Successfully uploaded /content/drive/MyDrive/data_of_RAG/The_Handmaiden.md with ID: file-SK3icvzJCuZEYAmqCCZeeN
Successfully uploaded /content/drive/MyDrive/data_of_RAG/Stoker.md with ID: file-TZURMmzk8Xb2nNSs2CSsqW
Successfully uploaded /content/drive/MyDrive/data_of_RAG/Film_noir.md with ID: file-89FRAgAzfAWwYqikk6LE6n
Creating file batch for Vector Store vs_6816b16854ec8191b6b5483ee0a6f8d9 with 5 uploaded files.
File Batch created with ID: vsfb_cb2564d73a2b4d4d8e530426fc245768, status: in_progress
Waiting for file batch processing to complete
File batch status: completed
File batch processed successfully. Files are now indexed in the vector stor

## Create conversation Thread

In [61]:
thread = client.beta.threads.create()
print(f"Conversation thread created: {thread.id}")

Conversation thread created: thread_tSrcxfXcHtYbylseZrpD3HFI


## Input your questions

question list:
- question 1: we can pick a few key scenes from Stoker together to analyse specifically where the emotional breaks are. For example, India's first murder, her dialogue with her mother, her uncle's time with her, etc
- question 2: Why the love story in Decision to Leave to be more moving for me than the love story in The Handmaiden.
- question 3: The Girl with the Dragon Tattoo with film noir




In [63]:
user_input = "tell me some movie directed by Park Chan-wook"

## Add user message for Thread

In [64]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content=user_input,
)

## Run Thread

In [65]:
print(f"\nStarting Run on thread {thread.id} with assistant {assistant.id}")
run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
    # override the assistant's instructions if needed
    # instructions="...",
)

print(f"Run created: {run.id}, status: {run.status}")

# Polling Run status and handling requires_action
print("Entering Run polling loop")
while run.status in ['queued', 'in_progress', 'cancelling', 'requires_action']:
    time.sleep(1) # Wait before checking status again

    run = client.beta.threads.runs.retrieve(
        thread_id=thread.id,
        run_id=run.id
    )
    print(f"Current Run status: {run.status}")

    # Calling Tools
    if run.status == 'requires_action':
        print("Assistant requires tool action Processing tool calls")
        tool_outputs = [] # List to collect results of tool calls for THIS requires_action step
        for tool_call in run.required_action.submit_tool_outputs.tool_calls:
            tool_call_id = tool_call.id # Unique ID for this tool call instance
            tool_type = tool_call.type # 'file_search' or 'function'
            print(f"Processing tool call ID: {tool_call_id}, Type: {tool_type}")
            # call function
            if tool_type == 'function':
                # Handle function calls
                function_name = tool_call.function.name # Name of the function requested
                function_args_str = tool_call.function.arguments # Arguments as a string

                print(f"Function Name: {function_name}")
                print(f"Function Arguments: {function_args_str}")

                # Find the corresponding Python function and execute it
                if function_name in available_functions:
                    function_to_call = available_functions[function_name]
                    try:
                        # Parse the arguments string into a Python dictionary
                        function_args = json.loads(function_args_str)

                        # Internal calls to the TMDB API
                        print(f"Executing local function: {function_name}...")
                        # Use ** to unpack the dictionary and pass arguments to the Python function
                        function_response_data = function_to_call(**function_args)
                        # Convert the function's return value to a JSON string
                        function_response_output = json.dumps(function_response_data)
                        print(f"Function execution complete. Output: {function_response_output}")

                    except json.JSONDecodeError:
                         # Handle cases where the assistant provides invalid JSON arguments
                         function_response_output = json.dumps({"error": f"Assistant provided invalid JSON args for {function_name}: {function_args_str}"})

                    except Exception as e:
                        function_response_output = json.dumps({"error": f"Error executing function {function_name}: {e}"})

                    # Add the result of this function call to the list of outputs
                    tool_outputs.append({
                        "tool_call_id": tool_call_id, # Must include the tool_call_id
                        "output": function_response_output, # The result of the function call (as a JSON string)
                    })

                else:
                    # If assistant requests a function not in available_functions
                    print(f"unknown function: {function_name}")
            else:
                print(f"Unknown tool type: {tool_type}")
                tool_outputs.append({
                    "tool_call_id": tool_call_id,
                    "output": json.dumps({"error": f"Unknown tool type {tool_type}"})
                })

        # Submitting Tool Output to the Assistants API
        if tool_outputs:
            print("Submitting tool outputs")
            try:
                 # Submit the list of tool outputs for this requires_action step
                 run = client.beta.threads.runs.submit_tool_outputs(
                     thread_id=thread.id,
                     run_id=run.id,
                     tool_outputs=tool_outputs
                 )
                 print(f"New Run status: {run.status}")
            # error catch
            except Exception as e:
                 print(f"tool_outputs: : {e}")
                 break
        else:
             print("tool_calls list was empty. Exiting loop.")
             break


    # final status
    # if run.status not in ['queued', 'in_progress', 'cancelling', 'requires_action']:
    #     print(f"Run finished with status: {run.status}. Exiting polling loop.")
    #     break

# Run End of loop

print(f"\n--- Final Run Status Check ---")
print(f"Run final status: {run.status}")

if run.status == 'completed':
    # Get and print the final reply message from the assistant
    messages = client.beta.threads.messages.list(
        thread_id=thread.id,
        order="asc", # Get messages in chronological order
        after=message.id
    )

    print("\n***** Assistant Response *****")
    if not messages.data:
        print("No new messages found from the assistant.")
    else:
        for msg in messages.data:
            if msg.role == "assistant": # Look for messages from the assistant
                # Messages can have multiple content blocks (text, image, tool output display)
                for content_block in msg.content:
                    if content_block.type == 'text':
                        print(content_block.text.value)
                        # Handle annotations for file citations or tool outputs display
                        if content_block.text.annotations:
                            print("Tool Info Below:")
                            for annotation in content_block.text.annotations:
                                if annotation.type == 'file_citation':
                                    # annotation.file_citation.file_id is OpenAI's internal file ID
                                    # annotation.file_citation.quote is the cited text snippet
                                    fc = annotation.file_citation

                                    # Old and new SDK compatibility: quote may not exist
                                    quote = getattr(fc, "quote", None) or getattr(annotation, "text", "")

                                    if quote:
                                        print(f"Quote: '{quote}'")
            # Optional: Loop through messages again to print role='tool' messages if needed
            # for msg in messages.data:
            #     if msg.role == 'tool':
            #          print(f"Tool Call ID: {msg.tool_call_id}")
            #          for content_block in msg.content:
            #             if content_block.type == 'text': # Tool output is text type content
            #                 print(f"  Output: {content_block.text.value}")


else:
    # Run did not complete successfully (failed, cancelled, expired)
    print(f"Run did not complete: {run.last_error or run.status}")



Starting Run on thread thread_tSrcxfXcHtYbylseZrpD3HFI with assistant asst_N1ZfjSzYrkv7nWQNbe1jZ0R4
Run created: run_mjWvUqYf2I65sDRjw16HQuOT, status: queued
Entering Run polling loop
Current Run status: queued
Current Run status: in_progress
Current Run status: in_progress
Current Run status: in_progress
Current Run status: in_progress
Current Run status: in_progress
Current Run status: in_progress
Current Run status: in_progress
Current Run status: in_progress
Current Run status: in_progress
Current Run status: completed

--- Final Run Status Check ---
Run final status: completed

***** Assistant Response *****
Here are some notable films directed by Park Chan-wook:

1. **Oldboy (2003)** - This film is part of Park's "Vengeance Trilogy" and is known for its intricate plot and stark portrayal of revenge, featuring exceptional performances and a memorable twist.

2. **The Handmaiden (2016)** - An adaptation of Sarah Waters's novel *Fingersmith*, this film interweaves themes of decepti

## The response of film assistant
### **question 1**
"Stoker," directed by Park Chan-wook, is a psychological thriller that intricately explores themes of grief, repression, and awakening through its protagonist, India Stoker. Here are a few key scenes that warrant close analysis for their emotional resonance:

### 1. India's First Murder
This pivotal scene marks a significant turning point in India's character development. The murder embodies her emotional awakening and shift from a passive observer to an active participant in her dark inherited impulses. The tension is palpable as she transitions from fear to exhilaration. The cinematography—crafted with stark contrasts and a meticulous color palette—enhances the psychological weight of the moment. The depiction of blood—richly symbolic in this context—underscores the awakening of her repressed desires. This experience opens the door to her dark potential and foreshadows her increasingly violent trajectory.

### 2. Dialogue with Her Mother, Evelyn
India’s interactions with her mother are laden with tension and unresolved emotion. Evelyn represents both a maternal figure and a source of oppression. Their conversations often reflect India's inner turmoil and isolation. Analyzing the subtext in their exchanges reveals how Evelyn's eerie calmness contrasts with India's burgeoning intensity. The dialogue is structured to reveal the emotional chasm between them, showcasing India's struggle for identity while trapped in her mother's controlling shadow. Ultimately, this dynamic complicates the theme of familial bonds and the weight of inheritance.

### 3. India's Relationship with Uncle Charlie
Charlie's arrival introduces a complex dynamic. His charm masks sinister undertones, and as India develops an attraction toward him, viewers witness a complex interplay of emotions—confusion, fear, intrigue. Analyze the moments when she observes him, allowing the cinematography to accentuate their chemistry and the underlying threat he poses. This relationship embodies themes of forbidden desire and moral ambiguity, raising questions about the nature of familial relationships and the dark allure of the unknown. India's evolving perception of Charlie serves as a reflection of her internal conflict, transitioning from admiration to apprehension to complicity.

### Emotional Breaks
In each of these scenes, the emotional breaks occur at critical junctures:
- **Realization** during India's first murder leads to an infusion of strength but raises questions about morality.
- **Vulnerability** during her dialogues with Evelyn highlights her internal conflict, showing the emotional burden of familial expectations.
- **Awakening** through her interactions with Charlie showcases her desire for liberation from her repressive environment but also the danger of indulgence in darkness.

In examining these scenes, "Stoker" brilliantly employs cinematography, sound design, and sharp dialogue to elevate the emotional stakes, painting a portrait of a young woman caught between the chains of her upbringing and her own burgeoning identity. Each moment serves as a building block for her transformation, making the exploration of these emotional breaks essential for understanding her character arc.
### **question 2**
Your emotional connection to the love story in *Decision to Leave* as more moving than that of *The Handmaiden* could stem from several key elements that resonate on a personal level. These elements include the nature of the protagonists' love, the emotional depth and moral complexity of their relationship, and the film's poignant portrayal of sacrifice and tragedy.

### 1. Emotional Depth and Complexity

In *Decision to Leave*, the love story between Detective Hae-jun and Seo-rae is fraught with emotional and moral complexity. Both characters are deeply flawed, facing their own loneliness and moral dilemmas. Hae-jun, despite his role as a detective, struggles with a stagnant marriage and a profound sense of isolation. Seo-rae, as a Chinese immigrant, carries the burdens of past trauma, including the suspicions surrounding her husband's death【8:0†source】【8:1†source】. This layered complexity evokes sympathy and a profound emotional connection, as viewers witness their struggles and hopes intermingle against a backdrop of suspicion and moral ambiguity.

The nature of their love is one where they are constantly separated by their respective circumstances—law and crime, duty and desire—which creates a tension that resonates with viewers. The moments of connection, such as sharing an umbrella in the rain or climbing mountains together, are imbued with an emotional weight that elevates their relationship to one of tragic yearning【8:1†source】【8:5†source】. This yearning is intensified by the film's tragic climax, where Seo-rae’s ultimate sacrifice—allowing herself to become an unsolved mystery to preserve Hae-jun's integrity—adds a haunting sense of beauty to their love【8:0†source】.

### 2. Tragic Romance

*Decision to Leave* employs a narrative approach that emphasizes tragedy over triumph. Unlike traditional love stories that often culminate in reunions or resolutions, this film concludes with a heartbreaking parting that underscores the pain of unattainable love. The film's final scenes linger in the mind, presenting a love story that is not only beautiful but also steeped in sorrow. The deep emotional resonance lies in the tragedy that unfolds—not necessarily a conventional happy ending, but one that reflects the complexities and sacrifices inherent in their love【8:0†source】【8:2†source】. This approach can evoke a more profound emotional response, as viewers are left to grapple with the implications of love's compromises.

### 3. Personal Reflection and Moral Dilemmas

Your connection to the love story in *Decision to Leave* may also be influenced by the film's exploration of moral dilemmas and human vulnerability. Hae-jun's inability to fully confront his feelings for Seo-rae, coupled with her secrets, creates a barrier that viewers can empathize with—a feeling of wanting to understand and forgive, even when faced with moral transgressions【8:4†source】【8:5†source】. This exploration of love intertwined with duty, suspicion, and ethical challenges resonates on a personal level, prompting reflections on how far one might go for love and the consequences that such choices entail.

### In Contrast: *The Handmaiden*

On the other hand, while *The Handmaiden* features a compelling love story, it primarily functions as a revenge thriller and liberation narrative. The emotional impact stems from themes of empowerment and social justice rather than the intricacies of romantic sacrifice. The resolution of *The Handmaiden* offers a sense of triumph, which can sometimes overshadow the intimate emotional experiences between the characters【8:15†source】【8:16†source】. This difference in narrative focus might not resonate as deeply on a personal level compared to the tragic and nuanced approach of *Decision to Leave*, especially if you find greater resonance in stories that emphasize longing, loss, and moral ambiguity.

### Conclusion

Ultimately, the moving nature of the love story in *Decision to Leave* may align more closely with themes that evoke sorrow, moral conflict, and tragic beauty. The emotional complexity, character isolation, and profound sacrifices depicted create an intensely moving experience that can linger long after viewing, appealing to a deeper exploration of the complexities of love and human experience. These elements combine to form a narrative that, for many viewers, feels more personally resonant than the love story presented in *The Handmaiden*.

### **question 3**
*The Girl with the Dragon Tattoo* can be examined through the lens of film noir, despite not being a traditional film noir. Here’s an analysis focusing on how it aligns with and diverges from the classic noir elements:

### Film Noir Characteristics in *The Girl with the Dragon Tattoo*

1. **Moral Ambiguity**:
   - Classic noir often features morally complex characters who navigate a shadowy world of deceit and corruption. Mikael Blomkvist and Lisbeth Salander fit this mold, as both operate in morally gray areas. Blomkvist is a disgraced journalist with a questionable ethical background, and Salander, while a victim of systemic abuse, employs methods that blur the lines of legality.

2. **Atmospheric Cinematography**:
   - David Fincher's direction captures a dark, moody atmosphere reminiscent of classic noir. The use of shadows, dim lighting, and urban settings creates an oppressive and tense environment that parallels the emotional and psychological weight of the story.

3. **Femme Fatale Archetype**:
   - Lisbeth Salander serves as a modern interpretation of the femme fatale, a staple of film noir. However, she subverts the trope; rather than being a manipulative seductress, she is a survivor and a determined avenger, wielding her intelligence and hacking skills as her primary weapons.

4. **Themes of Isolation and Betrayal**:
   - Both traditional film noir and *The Girl with the Dragon Tattoo* explore themes of isolation and betrayal. Blomkvist’s investigation reveals deep-seated familial betrayal and corporate corruption, and Salander's backstory emphasizes her isolation and the betrayal she has faced throughout her life.

5. **Complex Narrative**:
   - The film's intricate plot, which weaves together multiple storylines, is characteristic of many noir films. The unfolding mystery involves both personal and systemic corruption, striking at divergent threads of society—corporate greed and family secrets.

6. **Psychological Tension**:
   - Much like classic noir films, the psychological depth of the characters adds to the tension. The exploration of Salander's traumatic past and Blomkvist’s moral dilemmas adds layers of psychological complexity typical of the noir style.

### Divergences from Traditional Noir

- **Resolution**: While many noir films often conclude with a sense of hopelessness and moral degradation, *The Girl with the Dragon Tattoo* offers a more complex resolution that leaves room for justice, albeit at a high personal cost.

- **Protagonist's Agency**: Traditional noir often features passive protagonists who are swept along by circumstances. In contrast, both Blomkvist and Salander demonstrate significant agency, actively driving the narrative forward.

### Conclusion

*The Girl with the Dragon Tattoo* embodies many elements of film noir, particularly in its exploration of dark themes, morally ambiguous characters, and stylistic choices. However, it also evolves the genre by offering a modern perspective on agency and justice, employing contemporary storytelling techniques that resonate within the framework of traditional noir.
